In [1]:
# %%
import json
import re
import core.utils as oa
from rapidfuzz import fuzz
import pandas as pd
import statistics
import os
import io
from pprint import pprint
import math 

import datetime
from numpyencoder import NumpyEncoder

from pathlib import Path
import core.similaritysearch as simsearch
import core.vectorsearch as vecsearch

# root directory path
ROOT = Path(os.getcwd()).resolve().parents[0]

/home/janosch/anaconda3/envs/ma_orgelpredigt/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[nltk_data] Downloading package stopwords to
[nltk_data]     /home/janosch/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Error loading german: Package 'german' not found in index


In [2]:
date = datetime.datetime.now().strftime("%y-%m-%d_%H:%M")

In [3]:
with open(ROOT / f"sermons_with_most_music.json", "r") as f:
    testsermons_most = json.load(f)

with open(ROOT / f"sermons_with_longest_music.json", "r") as f:
    testsermons_longest = json.load(f)

In [4]:
relevant_page_texts = []
for n in range(41, 1291):
    with open(ROOT / f"source_texts/praxis_pietatis_verses/{n}.json") as f:
        page = json.load(f)
    page_info = {}
    page_info[n] = page
    relevant_page_texts.append(page_info)

In [5]:
def compare_with_validation(guessed_hits: pd.DataFrame, known_hits: pd.DataFrame, all_sents: int) -> dict:

    converged_df = pd.merge(known_hits, guessed_hits, on=['Paragraph','Satz'], how='inner')
    converged_df["in_page_list"]  = converged_df.apply(lambda row: row['Fundstelle'] in row['Ref_Seite'], axis=1)

    # analysis per verse
    val_hits_verse = len(known_hits)
    confirmed_true_negatives = all_sents - val_hits_verse
    
    merged_df = pd.merge(guessed_hits, known_hits, on=['Paragraph', 'Satz'], how='left', indicator=True)
    hits_not_in_val_verse = len(merged_df[merged_df['_merge'] == 'left_only'].drop('_merge', axis=1))
    
    agreed_hits_verse = converged_df["in_page_list"].value_counts()[True] # true pos
    divergent_hits_verse = len(converged_df) - agreed_hits_verse    # false pos
    missed_hits_verse = len(known_hits) - (agreed_hits_verse + divergent_hits_verse) # false neg
    avg_certainty = guessed_hits["Ähnlichkeit"].mean()

    true_negatives = all_sents - len(guessed_hits)

    tp = agreed_hits_verse
    tn = true_negatives
    fp = hits_not_in_val_verse
    fn = missed_hits_verse

    precision_verse = tp / (tp + fp)
    recall_verse = agreed_hits_verse / (tp + fn)

    f1_verse = (2 * precision_verse * recall_verse) / (precision_verse + recall_verse)

    accuracy_verse = (agreed_hits_verse + (all_sents - (agreed_hits_verse + divergent_hits_verse + missed_hits_verse))) / all_sents

    mc_verse = (tp * tn - fp * fn) / math.sqrt((tp + fp) * (tp + fn) * (tn + fp) * (tn + fn))

    # analysis per hit
    grouped_classifications = guessed_hits.copy().groupby(["Paragraph", "Fundstelle"])
    nr_of_classifications = len(list(grouped_classifications.groups.keys()))

    grouped_known_hits = known_hits.copy().groupby(["Referenz", "Paragraph"])
    val_hits = len(list(grouped_known_hits.groups.keys()))

    new_hits = merged_df[merged_df['_merge'] == 'left_only'].drop('_merge', axis=1)
    grouped_new_hits = new_hits.copy().groupby(["Paragraph", "Fundstelle"])
    hits_not_in_val = len(list(grouped_new_hits.groups.keys()))

    grouped_hits = converged_df.copy().groupby(["Referenz", "Paragraph"])
    group_keys = list(grouped_hits.groups.keys())

    page_matches = 0
    page_mismatches = 0

    for name, group in grouped_hits:
        known_pages = group['Ref_Seite'].iloc[0]
        guessed_pages = group["Fundstelle"].to_list()
        
        if len(set(known_pages).intersection(guessed_pages)) > 0:
            page_matches += 1
        else: 
            page_mismatches += 1
    
    agreed_hits = page_matches
    divergent_hits = page_mismatches
    missed_hits = val_hits - (agreed_hits + divergent_hits)

    precision_hits = agreed_hits / (agreed_hits + divergent_hits + hits_not_in_val)
    recall_hits = agreed_hits / val_hits

    f1_hits = (2 * precision_hits * recall_hits) / (precision_hits + recall_hits)

    results = {}

    results["id"] = id
    results["identified_hits_total"] = nr_of_classifications
    results["song_quotes_total"] = val_hits
    results["sentences total"] = all_sents
    results["verse_agreed_hits"] = agreed_hits_verse
    results["verse_divergent_hits"] = divergent_hits_verse
    results["verse_new_hits"] = hits_not_in_val_verse
    results["verse_missed_hits"] = missed_hits_verse
    results["verse_avg_certainty"] = avg_certainty
    results["verse_matthews_coeff"] = mc_verse

    results["verse_precision"] = precision_verse
    results["verse_recall"] = recall_verse
    results["verse_f1-score"] = f1_verse
    results["verse_accuracy"] = accuracy_verse

    results["hits_agreed"] = agreed_hits
    results["hits_divergent"] = divergent_hits
    results["hits_new"] = hits_not_in_val
    results["hits_missed"] = missed_hits

    results["hits_precision"] = precision_hits
    results["hits_recall"] = recall_hits
    results["hits_f1-score"] = f1_hits

    return results

In [6]:
def compute_averages(nested_list: list) -> list:

    if not nested_list:
        return []

    averages = []

    # Initialize a list to store the sums of the last 5 elements
    sums = [0.0] * 5

    first_six = nested_list[0][:6]

    for sublist in nested_list:
        for i in range(5):
            sums[i] += sublist[6 + i]

     # Calculate the averages
    averages = [round(s / len(nested_list), 4) for s in sums]

    # Combine the first 6 elements and the averages
    result = first_six + averages

    return result

# Similarity Search

In [15]:
all_results = []
for fuzz in [83]:
    print(f"starting with fuzziness {fuzz}")

    # 1: just remove dupl, 2: add inf.m., 3: corr.inbtw. m.
    for setup in [3]:#, 2, 3]:
        print(f"starting on {setup}")
        single_results = []
        for id in testsermons_most:

            sermon = oa.Sermon(id)

            # get total number of sentences
            total_sentences = 0
            for paragraph in sermon.chunked:
                total_sentences += len(paragraph)

            # create validation set
            validation = []
            for i in range(len(sermon.chunked)):                # for each paragraph
                for j in range(len(sermon.chunked[i])):         # for each sentence
                    if " musikwerk" in sermon.chunked[i][j]["types"]:
                        line = " ".join(sermon.chunked[i][j]["words"])
                        refs = ", ".join(set(simsearch.flatten(sermon.chunked[i][j]["references"])))
                        validation.append([line, i, j, refs])

            known_hits = pd.DataFrame(validation, columns=["Predigt", "Paragraph", "Satz", "Referenz"])
            known_hits = known_hits[known_hits['Referenz'].apply(simsearch.is_song_in_book)]
            known_hits["Ref_Seite"] = known_hits['Referenz'].apply(simsearch.song_page)

            guessed_hits = simsearch.find_similarities("lieder", id, relevant_page_texts, fuzz, test=True)

            # test only with remove_duplicates
            if setup == 1:
                comparison = compare_with_validation(guessed_hits, 
                                                     known_hits, 
                                                     total_sentences)

                single_sermon_result = ["5 Predigten mit längsten Liedzitaten", "Ähnlichkeitssuche", fuzz, "Ja", "Nein", "Nein", comparison["verse_f1-score"], comparison["verse_precision"], comparison["verse_recall"], comparison["verse_matthews_coeff"], comparison["verse_avg_certainty"]]
                single_results.append(single_sermon_result)
            # test with remove_duplicates and add inferred matches
            elif setup == 2:
                guessed_hits = simsearch.add_inferred_matches(guessed_hits, id)
                comparison = compare_with_validation(guessed_hits, 
                                                     known_hits, 
                                                     total_sentences)

                single_sermon_result = ["5 Predigten mit längsten Liedzitaten", "Ähnlichkeitssuche", fuzz, "Ja", "Ja", "Nein", comparison["verse_f1-score"], comparison["verse_precision"], comparison["verse_recall"], comparison["verse_matthews_coeff"], comparison["verse_avg_certainty"]]
                single_results.append(single_sermon_result)
            # test with remove_duplicates, add inferred matches and correct inbetween matches
            else:
                guessed_hits = simsearch.add_inferred_matches(guessed_hits, id)
                guessed_hits = simsearch.correct_inbetween_matches(guessed_hits)
                comparison = compare_with_validation(guessed_hits, 
                                                     known_hits, 
                                                     total_sentences)

                single_sermon_result = ["5 Predigten mit längsten Liedzitaten", "Ähnlichkeitssuche", fuzz, "Ja", "Ja", "Ja", comparison["verse_f1-score"], comparison["verse_precision"], comparison["verse_recall"], comparison["verse_matthews_coeff"], comparison["verse_avg_certainty"]]
                single_results.append(single_sermon_result)
        
        # create averages out of the 5 individual results
        avg_values = compute_averages(single_results)
        all_results.append(avg_values)

results_df = pd.DataFrame(all_results, columns=["Korpus", "Methode", "Unschärfefaktor", "Dopplungen entfernt", "Treffer inferiert", "Treffer korrigiert", "F1-Score", "Precision", "Recall", "Matthews-Coefficient", "Sicherheit"])


starting with fuzziness 83
starting on 3
Starting with E000036
Starting with E000072
Starting with E000070


/home/janosch/Projects/Personal/ma-project/core/similaritysearch.py:131: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  guessed_hits = pd.concat([guessed_hits, new_matches])


Starting with E000055


/home/janosch/Projects/Personal/ma-project/core/similaritysearch.py:131: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  guessed_hits = pd.concat([guessed_hits, new_matches])
/home/janosch/Projects/Personal/ma-project/core/similaritysearch.py:131: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  guessed_hits = pd.concat([guessed_hits, new_matches])


Starting with E000042


/home/janosch/Projects/Personal/ma-project/core/similaritysearch.py:131: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  guessed_hits = pd.concat([guessed_hits, new_matches])


In [16]:
results_df

,Korpus,Methode,Unschärfefaktor,Dopplungen entfernt,Treffer inferiert,Treffer korrigiert,F1-Score,Precision,Recall,Matthews-Coefficient,Sicherheit
0,5 Predigten mit längsten Liedzitaten,Ähnlichkeitssuche,83,Ja,Ja,Ja,0.6257,0.6683,0.6117,0.6254,87.8946


In [13]:
results_df

,Korpus,Methode,Unschärfefaktor,Dopplungen entfernt,Treffer inferiert,Treffer korrigiert,F1-Score,Precision,Recall,Matthews-Coefficient,Sicherheit
0,5 Predigten mit längsten Liedzitaten,Ähnlichkeitssuche,75,Ja,Nein,Nein,0.4294,0.3540,0.6750,0.4489,84.2762
1,5 Predigten mit längsten Liedzitaten,Ähnlichkeitssuche,80,Ja,Nein,Nein,0.5381,0.5460,0.5744,0.5385,89.5274
2,5 Predigten mit längsten Liedzitaten,Ähnlichkeitssuche,83,Ja,Nein,Nein,0.5768,0.6757,0.5200,0.5792,92.2557
3,5 Predigten mit längsten Liedzitaten,Ähnlichkeitssuche,90,Ja,Nein,Nein,0.5110,0.8269,0.3766,0.5474,96.1567


In [ ]:
results_no_optimisation_bible_not_filtered

,Korpus,Methode,Unschärfefaktor,Dopplungen entfernt,Treffer inferiert,Treffer korrigiert,F1-Score,Precision,Recall,Matthews-Coefficient,Sicherheit
0,5 Predigten mit meisten Liedern,Ähnlichkeitssuche,75,Ja,Nein,Nein,0.3469,0.2562,0.6771,0.3757,83.5105
1,5 Predigten mit meisten Liedern,Ähnlichkeitssuche,80,Ja,Nein,Nein,0.4685,0.4221,0.5776,0.4697,88.9933
2,5 Predigten mit meisten Liedern,Ähnlichkeitssuche,83,Ja,Nein,Nein,0.5212,0.5403,0.5232,0.5156,91.9306
3,5 Predigten mit meisten Liedern,Ähnlichkeitssuche,90,Ja,Nein,Nein,0.4851,0.6889,0.3803,0.5005,95.9597


In [22]:
results_df

,Korpus,Methode,Unschärfefaktor,Dopplungen entfernt,Treffer inferiert,Treffer korrigiert,F1-Score,Precision,Recall,Matthews-Coefficient,Sicherheit
0,5 Predigten mit meisten Liedern,Ähnlichkeitssuche,82,True,False,False,0.6389,0.7775,0.5457,0.6357,93.0469
1,5 Predigten mit meisten Liedern,Ähnlichkeitssuche,82,True,True,False,0.7175,0.7885,0.6628,0.7089,88.5106
2,5 Predigten mit meisten Liedern,Ähnlichkeitssuche,82,True,True,True,0.7197,0.7906,0.6651,0.7111,88.3818
3,5 Predigten mit meisten Liedern,Ähnlichkeitssuche,83,True,False,False,0.6416,0.8154,0.5326,0.6439,93.7147
4,5 Predigten mit meisten Liedern,Ähnlichkeitssuche,83,True,True,False,0.7248,0.8215,0.6541,0.7193,88.7889
5,5 Predigten mit meisten Liedern,Ähnlichkeitssuche,83,True,True,True,0.7282,0.8246,0.6578,0.7227,88.6557
6,5 Predigten mit meisten Liedern,Ähnlichkeitssuche,84,True,False,False,0.6268,0.8331,0.5064,0.6348,94.4971
7,5 Predigten mit meisten Liedern,Ähnlichkeitssuche,84,True,True,False,0.7060,0.8340,0.6197,0.7047,89.3783
8,5 Predigten mit meisten Liedern,Ähnlichkeitssuche,84,True,True,True,0.7092,0.8365,0.6232,0.7078,89.2390


In [20]:
results_80_82_85

,Korpus,Methode,Unschärfefaktor,Dopplungen entfernt,Treffer inferiert,Treffer korrigiert,F1-Score,Precision,Recall,Matthews-Coefficient,Sicherheit
0,5 Predigten mit meisten Liedern,Ähnlichkeitssuche,80,True,False,False,0.6330,0.6623,0.6065,0.6160,90.4069
1,5 Predigten mit meisten Liedern,Ähnlichkeitssuche,80,True,True,False,0.6990,0.6782,0.7216,0.6836,86.6455
2,5 Predigten mit meisten Liedern,Ähnlichkeitssuche,80,True,True,True,0.7014,0.6808,0.7238,0.6860,86.5335
3,5 Predigten mit meisten Liedern,Ähnlichkeitssuche,82,True,False,False,0.6389,0.7775,0.5457,0.6357,93.0469
4,5 Predigten mit meisten Liedern,Ähnlichkeitssuche,82,True,True,False,0.7175,0.7885,0.6628,0.7089,88.5106
5,5 Predigten mit meisten Liedern,Ähnlichkeitssuche,82,True,True,True,0.7197,0.7906,0.6651,0.7111,88.3818
6,5 Predigten mit meisten Liedern,Ähnlichkeitssuche,85,True,False,False,0.6240,0.8392,0.5006,0.6333,94.8340
7,5 Predigten mit meisten Liedern,Ähnlichkeitssuche,85,True,True,False,0.6896,0.8350,0.5959,0.6904,90.3338
8,5 Predigten mit meisten Liedern,Ähnlichkeitssuche,85,True,True,True,0.6917,0.8367,0.5984,0.6925,90.1907


In [16]:
old_results

,Korpus,Methode,Unschärfefaktor,Dopplungen entfernt,Treffer inferiert,Treffer korrigiert,F1-Score,Precision,Recall,Matthews-Coefficient,Sicherheit
0,5 Predigten mit meisten Liedern,Ähnlichkeitssuche,90,True,False,False,0.5917,0.9008,0.4453,0.6187,96.6387
1,5 Predigten mit meisten Liedern,Ähnlichkeitssuche,90,True,True,False,0.6707,0.8957,0.5424,0.6832,91.5294
2,5 Predigten mit meisten Liedern,Ähnlichkeitssuche,90,True,True,True,0.6732,0.8971,0.5455,0.6856,91.4438
3,5 Predigten mit meisten Liedern,Ähnlichkeitssuche,80,True,False,False,0.6330,0.6623,0.6065,0.6160,90.4069
4,5 Predigten mit meisten Liedern,Ähnlichkeitssuche,80,True,True,False,0.6990,0.6782,0.7216,0.6836,86.6455
5,5 Predigten mit meisten Liedern,Ähnlichkeitssuche,80,True,True,True,0.7014,0.6808,0.7238,0.6860,86.5335
6,5 Predigten mit meisten Liedern,Ähnlichkeitssuche,70,True,False,False,0.3697,0.2439,0.7640,0.3906,78.9221
7,5 Predigten mit meisten Liedern,Ähnlichkeitssuche,70,True,True,False,0.3354,0.2071,0.8818,0.3832,71.7357
8,5 Predigten mit meisten Liedern,Ähnlichkeitssuche,70,True,True,True,0.3379,0.2090,0.8827,0.3852,71.7006


## Ergebnis des ersten Tests anhand der 5 Predigten mit den meisten Zitaten aus Musikwerken:
- Der erste Testdatensatz bestand aus den 5 Predigten mit den _meisten_ zitierten Musikwerken:
```
[{'id': 'E000036',
  'name': 'Raphael Jonathan Skubowius,  ()- (): Die heilige Sabbaths-Lust an dem Herrn (Danzig 1749)',
  'words': 796,
  'Prozentsatz': 5.74024662868681,
  'zitierte_werke': 25},
 {'id': 'E000072',
  'name': 'Christoph Friedrich Bucher, 1651/12/30 (Zabeltitz)-1716/03/24 (Rengersdorf): Gott und Gnug (Meißen 1681)',
  'words': 607,
  'Prozentsatz': 4.155541863490107,
  'zitierte_werke': 18},
 {'id': 'E000042',
  'name': 'Christian Friedrich Wilisch, 1684/09/21 (Liebstadt)-1759/01/02 (Freiberg): Das Neue Lied (Freiberg 1735)',
  'words': 190,
  'Prozentsatz': 1.2779122948614474,
  'zitierte_werke': 17},
 {'id': 'E000070',
  'name': 'Johann Christoph Thiele, 1637/06/14 (Fröttstädt)-1710/09/11 (Effelder): Cithara Theologica (Schleusingen 1683)',
  'words': 420,
  'Prozentsatz': 4.168320762207225,
  'zitierte_werke': 16},
 {'id': 'E000055',
  'name': 'Gabriel Hanitsch, 1673/10/26 (Glashütte)-1736 (Naundorf): Davids Vermahnung (Dresden 1711)',
  'words': 153,
  'Prozentsatz': 1.028778913394298,
  'zitierte_werke': 15}]
```
- In diesen Predigten nehmen Liedzitate zwischen 5,74% und 1,02% des gesamttextes ein.

- Die Ergebnisse waren entsprechend durchmischt: die Besten resultate wurden mit einem fuzziness-Faktor von 83 erziehlt (F1-score: 0.5212, Matthews-Coefficient: 0.5156):
F1: 0.5212, Precision: 0.5403, Recall: 0.5232, Matthews-Coefficient: 0.5156, Avg. Certainty: 91.9306
- Die beiden Postprocessing-Schritte konnten die Ergebnisse wie folgt verbessern:
F1: 0.5626, Precision: 0.5356, Recall: 0.6157, Matthews-Coefficient: 0.5587, Avg. Certainty: 87.6327
- Hier wurde der F1-score um 4,14% verbessert.
- Die Ergebnisse sind _deutlich schlechter_ als bei den vorhergehenden Tests anhand der beiden Predigten [E000036, E000052]: Bei gleicher Fuzziness und Postprocessing:
F1: 0.7282, Precision: 0.8246, Recall: 0.6578, Matthews-Coefficient: 0.7227, Avg. Certainty: 88.6557
- Dabei wurde der F1-Score um 8,6% verbessert
- Beim Testen mit vorherigem Filtern der bekannten Bibelzitate konnte folgende Steigerung erziehlt werden:
F1: 0.5768, Precision: 0.6757, Recall: 0.52, Matthews-Coefficient: 0.5792, Avg. Certainty: 92.2557
- Mit Postprocessing ergibt das:
F1: 0.6257, Precision: 0.6683, Recall: 0.6117, Matthews-Coefficient: 0.6254, Avg. Certainty: 87.8946
- Der F1-Score wird hier beim post-processing um 4,89% verbessert.
- Die Diskrepanz zwischen Bibelzitate entfernt und belassen beträgt somit 6,31% (F1) bzw. 6,67% (Matthews-Coefficient)

# Vector Search

In [17]:
from typing import List
from sentence_transformers import SentenceTransformer
from langchain_community.vectorstores import Chroma
from langchain_core.documents.base import Document
from langchain_core.embeddings.embeddings import Embeddings
from langchain_core.runnables import chain
import core.vectorsearch as vecsearch

In [18]:
model_name = "LaBSE"

In [11]:
model = SentenceTransformer(f'sentence-transformers/{model_name}')


class EmbedSomething(Embeddings):
    def __init__(self,model) -> None:
        self.model = model

    def embed_documents(self,texts):
        t = self.model.encode(texts)
        return t.tolist()

    def embed_query(self, text: str) -> List[float]:
        t = self.model.encode(text)
        return t.tolist()

emb = EmbedSomething(model)

directory = str(ROOT / f"./chroma/chroma_db_{model_name}")
vectordb = Chroma(persist_directory = directory, embedding_function=emb)

@chain
def retriever(inputs: dict) -> tuple[Document]:
    query = inputs["query"]
    page = inputs.get("page")
    filter_criteria = {}
    if page:
        filter_criteria["source"] = str(page)
    if not filter_criteria:
        filter_criteria = None

    docs, scores = zip(
        *vectordb.similarity_search_with_score(
            query,
            k=1,
            filter=filter_criteria
        )
    )
    for doc, score in zip(docs, scores):
        doc.metadata["score"] = score
    return docs

/tmp/ipykernel_168775/2558610308.py:19: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-chroma package and should be used instead. To use it run `pip install -U :class:`~langchain-chroma` and import as `from :class:`~langchain_chroma import Chroma``.
  vectordb = Chroma(persist_directory = directory, embedding_function=emb)


In [52]:
all_results = []
for fuzz in [40]:
    cosine_cutoff = (100 - fuzz) * 0.01
    print(f"starting with fuzziness {fuzz}")

    # 1: just remove dupl, 2: add inf.m., 3: corr.inbtw. m.
    for setup in [3]:#, 2, 3]:
        print(f"starting on {setup}")
        single_results = []
        for id in testsermons_most:

            sermon = oa.Sermon(id)

            # get total number of sentences
            total_sentences = 0
            for paragraph in sermon.chunked:
                total_sentences += len(paragraph)

            # create validation set
            validation = []
            for i in range(len(sermon.chunked)):                # for each paragraph
                for j in range(len(sermon.chunked[i])):         # for each sentence
                    if " musikwerk" in sermon.chunked[i][j]["types"]:
                        line = " ".join(sermon.chunked[i][j]["words"])
                        refs = ", ".join(set(vecsearch.flatten(sermon.chunked[i][j]["references"])))
                        validation.append([line, i, j, refs])

            known_hits = pd.DataFrame(validation, columns=["Predigt", "Paragraph", "Satz", "Referenz"])
            known_hits = known_hits[known_hits['Referenz'].apply(vecsearch.is_song_in_book)]
            known_hits["Ref_Seite"] = known_hits['Referenz'].apply(vecsearch.song_page)

            guessed_hits = vecsearch.find_similarities("lieder", id, cosine_cutoff, retriever, test=False)

            # test only with remove_duplicates
            if setup == 1:
                comparison = compare_with_validation(guessed_hits, 
                                                     known_hits, 
                                                     total_sentences)

                single_sermon_result = ["5 Predigten mit meisten Liedzitaten", "Vektorsuche", cosine_cutoff, "Ja", "Nein", "Nein", comparison["verse_f1-score"], comparison["verse_precision"], comparison["verse_recall"], comparison["verse_matthews_coeff"], comparison["verse_avg_certainty"]]
                single_results.append(single_sermon_result)
            # test with remove_duplicates and add inferred matches
            elif setup == 2:
                guessed_hits = vecsearch.add_inferred_matches(guessed_hits, id, retriever)
                comparison = compare_with_validation(guessed_hits, 
                                                     known_hits, 
                                                     total_sentences)

                single_sermon_result = ["5 Predigten mit meisten Liedzitaten", "Vektorsuche", cosine_cutoff, "Ja", "Ja", "Nein", comparison["verse_f1-score"], comparison["verse_precision"], comparison["verse_recall"], comparison["verse_matthews_coeff"], comparison["verse_avg_certainty"]]
                single_results.append(single_sermon_result)
            # test with remove_duplicates, add inferred matches and correct inbetween matches
            else:
                guessed_hits = vecsearch.add_inferred_matches(guessed_hits, id, retriever)
                guessed_hits = vecsearch.correct_inbetween_matches(guessed_hits, retriever)
                comparison = compare_with_validation(guessed_hits, 
                                                     known_hits, 
                                                     total_sentences)

                single_sermon_result = ["5 Predigten mit meisten Liedzitaten", "Vektorsuche", cosine_cutoff, "Ja", "Ja", "Ja", comparison["verse_f1-score"], comparison["verse_precision"], comparison["verse_recall"], comparison["verse_matthews_coeff"], comparison["verse_avg_certainty"]]
                single_results.append(single_sermon_result)
        
        # create averages out of the 5 individual results
        avg_values = compute_averages(single_results)
        all_results.append(avg_values)

results_df = pd.DataFrame(all_results, columns=["Korpus", "Methode", "Unschärfefaktor", "Dopplungen entfernt", "Treffer inferiert", "Treffer korrigiert", "F1-Score", "Precision", "Recall", "Matthews-Coefficient", "Sicherheit"])

starting with fuzziness 40
starting on 3
starting with E000036
starting with E000072


/home/janosch/Projects/Personal/ma-project/core/vectorsearch.py:150: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  guessed_hits = pd.concat([guessed_hits, new_matches])
/home/janosch/Projects/Personal/ma-project/core/vectorsearch.py:150: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  guessed_hits = pd.concat([guessed_hits, new_matches])


starting with E000070


/home/janosch/Projects/Personal/ma-project/core/vectorsearch.py:150: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  guessed_hits = pd.concat([guessed_hits, new_matches])


starting with E000055


/home/janosch/Projects/Personal/ma-project/core/vectorsearch.py:150: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  guessed_hits = pd.concat([guessed_hits, new_matches])
/home/janosch/Projects/Personal/ma-project/core/vectorsearch.py:150: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  guessed_hits = pd.concat([guessed_hits, new_matches])


starting with E000042


/home/janosch/Projects/Personal/ma-project/core/vectorsearch.py:150: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  guessed_hits = pd.concat([guessed_hits, new_matches])


In [51]:
results_df

,Korpus,Methode,Unschärfefaktor,Dopplungen entfernt,Treffer inferiert,Treffer korrigiert,F1-Score,Precision,Recall,Matthews-Coefficient,Sicherheit
0,5 Predigten mit meisten Liedzitaten,Vektorsuche,0.65,Ja,Nein,Nein,0.4032,0.3557,0.5489,0.4081,0.2598
1,5 Predigten mit meisten Liedzitaten,Vektorsuche,0.60,Ja,Nein,Nein,0.4109,0.3639,0.5489,0.4149,0.2477
2,5 Predigten mit meisten Liedzitaten,Vektorsuche,0.55,Ja,Nein,Nein,0.4090,0.3682,0.5215,0.4092,0.2277
3,5 Predigten mit meisten Liedzitaten,Vektorsuche,0.50,Ja,Nein,Nein,0.3931,0.3664,0.4795,0.3908,0.2083
4,5 Predigten mit meisten Liedzitaten,Vektorsuche,0.45,Ja,Nein,Nein,0.3934,0.3785,0.4621,0.3904,0.1869
5,5 Predigten mit meisten Liedzitaten,Vektorsuche,0.40,Ja,Nein,Nein,0.4045,0.4103,0.4435,0.4004,0.1558
6,5 Predigten mit meisten Liedzitaten,Vektorsuche,0.35,Ja,Nein,Nein,0.3526,0.3792,0.3560,0.3446,0.1356
7,5 Predigten mit meisten Liedzitaten,Vektorsuche,0.30,Ja,Nein,Nein,0.3162,0.3581,0.3098,0.3095,0.1212
8,5 Predigten mit meisten Liedzitaten,Vektorsuche,0.25,Ja,Nein,Nein,0.3115,0.3615,0.2993,0.3057,0.1135
9,5 Predigten mit meisten Liedzitaten,Vektorsuche,0.20,Ja,Nein,Nein,0.2921,0.3531,0.2711,0.2868,0.1080


In [47]:
results_df

,Korpus,Methode,Unschärfefaktor,Dopplungen entfernt,Treffer inferiert,Treffer korrigiert,F1-Score,Precision,Recall,Matthews-Coefficient,Sicherheit
0,5 Predigten mit meisten Liedzitaten,Vektorsuche,45,Ja,Ja,Ja,0.5268,0.5255,0.5943,0.5307,0.2677


In [42]:
results_45_50

,Korpus,Methode,Unschärfefaktor,Dopplungen entfernt,Treffer inferiert,Treffer korrigiert,F1-Score,Precision,Recall,Matthews-Coefficient,Sicherheit
0,5 Predigten mit meisten Liedzitaten,Vektorsuche,45,Ja,Nein,Nein,0.4893,0.5119,0.5175,0.4892,0.2206
1,5 Predigten mit meisten Liedzitaten,Vektorsuche,50,Ja,Nein,Nein,0.4721,0.5293,0.4756,0.4750,0.1957


In [37]:
results_55_80

,Korpus,Methode,Unschärfefaktor,Dopplungen entfernt,Treffer inferiert,Treffer korrigiert,F1-Score,Precision,Recall,Matthews-Coefficient,Sicherheit
0,5 Predigten mit meisten Liedzitaten,Vektorsuche,55,Ja,Nein,Nein,0.4727,0.5573,0.4582,0.4785,0.1712
1,5 Predigten mit meisten Liedzitaten,Vektorsuche,60,Ja,Nein,Nein,0.4671,0.5829,0.4398,0.4776,0.1501
2,5 Predigten mit meisten Liedzitaten,Vektorsuche,65,Ja,Nein,Nein,0.4113,0.5507,0.3560,0.4204,0.1313
3,5 Predigten mit meisten Liedzitaten,Vektorsuche,70,Ja,Nein,Nein,0.3631,0.5255,0.3098,0.3777,0.1125
4,5 Predigten mit meisten Liedzitaten,Vektorsuche,75,Ja,Nein,Nein,0.3528,0.5191,0.2993,0.3681,0.1080
5,5 Predigten mit meisten Liedzitaten,Vektorsuche,80,Ja,Nein,Nein,0.3302,0.5117,0.2711,0.3474,0.1025
6,5 Predigten mit meisten Liedzitaten,Vektorsuche,85,Ja,Nein,Nein,0.2817,0.5948,0.1938,0.3223,0.0637


In [45]:
all_no_opt = pd.concat([results_df, results_45_50, results_55_80])
all_no_opt

,Korpus,Methode,Unschärfefaktor,Dopplungen entfernt,Treffer inferiert,Treffer korrigiert,F1-Score,Precision,Recall,Matthews-Coefficient,Sicherheit
0,5 Predigten mit meisten Liedzitaten,Vektorsuche,35,Ja,Nein,Nein,0.4751,0.4725,0.5450,0.4771,0.2595
1,5 Predigten mit meisten Liedzitaten,Vektorsuche,40,Ja,Nein,Nein,0.4827,0.4818,0.5450,0.4839,0.2497
0,5 Predigten mit meisten Liedzitaten,Vektorsuche,45,Ja,Nein,Nein,0.4893,0.5119,0.5175,0.4892,0.2206
1,5 Predigten mit meisten Liedzitaten,Vektorsuche,50,Ja,Nein,Nein,0.4721,0.5293,0.4756,0.4750,0.1957
0,5 Predigten mit meisten Liedzitaten,Vektorsuche,55,Ja,Nein,Nein,0.4727,0.5573,0.4582,0.4785,0.1712
1,5 Predigten mit meisten Liedzitaten,Vektorsuche,60,Ja,Nein,Nein,0.4671,0.5829,0.4398,0.4776,0.1501
2,5 Predigten mit meisten Liedzitaten,Vektorsuche,65,Ja,Nein,Nein,0.4113,0.5507,0.3560,0.4204,0.1313
3,5 Predigten mit meisten Liedzitaten,Vektorsuche,70,Ja,Nein,Nein,0.3631,0.5255,0.3098,0.3777,0.1125
4,5 Predigten mit meisten Liedzitaten,Vektorsuche,75,Ja,Nein,Nein,0.3528,0.5191,0.2993,0.3681,0.1080
5,5 Predigten mit meisten Liedzitaten,Vektorsuche,80,Ja,Nein,Nein,0.3302,0.5117,0.2711,0.3474,0.1025


## Ergebnisse aus den Vektortests
### 1. 5 Predigten mit den meisten Liedzitaten, ohne Postprocessing, ohne Bibelzitate
- Bei Verwendung der Cosinus-Ähnlichkeit äquivalente zur Fuzziness (0.25,0.2,0.17,0.1) sind die Ergebnisse deutlich schlechter, das beste Ergebnis erziehlt die Cosinus-Ähnlichkeit 0.25:
F1: 0.3528, Prec: 0.5191, Recall: 0.2993, Matthews-Coefficient: 0.3681, Avg. Cosine Similarity: 0.108
- Bei Anwendung der Postprocessing-Schritte ergibt das:
F1: 0.4135, Prec: 0.5487, Recall: 0.3862, Mattews-Coefficient: 0.4286, Avg. Cosine Similarity: 0.1876
-> eine Verbesserung von 6,07%

- Bei Cosinus-Ähnlichkeit von 0,2 bis 0,55 werden die Ergebnisse kontinuierlich besser, bei noch größeren Werten nehmen sie wieder ab. Bestes ergebnis:
F1: 0.4893, Prec: 0.5119, Recall: 0.5175, MCC: 0.4892, Avg. Cosine Similarity: 0.2206
- Postprocessing ergibt folgende Verbesserung:
F1: 0.5268, Prec: 0.5255, Recall: 0.5943, MCC: 0.5307, Avg. Cosine Similarity: 0.2677
-> F1- Verbesserung von 3,75%

- Bei nicht-herausfiltern der Bibelzitate performt cosinus similarity = 0.6 am besten:
F1: 0.4109, Prec: 0.3639, Recall: 0.5489, MCC: 0.4149, Avg. Cosine Similarity: 0.2477
